# Multiple instance learning on patients

A label such as a diagnosis describes a patient, while the measurements are cells. Multiple instance learning treats
the cells of a patient as one bag with one label. {class}`~hidata.ml.MILDataset` yields such a bag for every group of
a HiData, and {class}`~hidata.ml.GroupMILModel` predicts the label from the features of the group and its cells: it
encodes both, pools the cells with attention and classifies the result.

This tutorial uses the COVID-19 cohort of the {doc}`patient cohort tutorial <patient_cohort>` and needs the `ml` extra:

```console
pip install "hidata[ml]"
```

In [1]:
import pandas as pd
import pytorch_lightning as pl
import scanpy as sc
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler

from hidata import HiData
from hidata.ml import GroupMILModel, MILDataset, mil_collate_fn

adata = sc.read(
    "data/stephenson_2021_subsampled.h5ad",
    backup_url="https://scverse-exampledata.s3.eu-west-1.amazonaws.com/pertpy/stephenson_2021_subsampled.h5ad",
)

## Patients as groups

Every patient is a group. The label is whether the patient has COVID-19. Sex and smoking status serve as patient
features, one-hot encoded in `obsm` of the group level, and the cells are described by their scVI embedding.

In [2]:
hd = HiData.from_anndata(adata, group_id="patient_id")
hd.G["covid"] = (hd.G["Status"] == "Covid").astype(int)
hd.obsm["clinical"] = pd.get_dummies(hd.G[["sex", "Smoker"]], dtype=float)
hd

╔═ HiData(n_groups=119, n_cells=62,509) ═══════════════════════════════════════╗
║ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ ║
║ ┃ G (groups)                         ┃ C (cells, 376..1,000 per group)     ┃ ║
║ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩ ║
║ │ DataFrame with 119 groups × 18     │ View of AnnData object with n_obs × │ ║
║ │ columns                            │ n_vars = 62,509 × 16,299            │ ║
║ │     columns: 'Swab_result',        │     obs: 'Collection_Day',          │ ║
║ │ 'Status', 'Smoker',                │ 'patient_id', 'author_cell_type',   │ ║
║ │ 'Status_on_day_collection',        │ 'cell_type'                         │ ║
║ │ 'Status_on_day_collection_summary… │                                     │ ║
║ │ 'Days_from_onset', 'Site',         │                                     │ ║
║ │ 'time_after_LPS',                  │                                     │ ║
║ │ 'Worst_Clinical_Status',           │                                     │ ║
║ │ 'Outcome', 'organism', 'sex',      │                                     │ ║
║ │ 'tissue', 'ethnicity', 'disease',  │                                     │ ║
║ │ 'assay', 'dataset_group', 'covid'  │                                     │ ║
║ │     obsm: 'clinical'               │     var: 'gene_id', 'gene_name'     │ ║
║ │                                    │     uns: 'log1p', 'umap'            │ ║
║ │                                    │     obsm: 'X_scVI', 'X_umap'        │ ║
║ └────────────────────────────────────┴─────────────────────────────────────┘ ║
╚═ group_id='patient_id' ══════════════════════════════════════════════════════╝

In [3]:
pd.crosstab(hd.G["Status"], hd.G["covid"])

covid,0,1
Status,,
Covid,0,90
Healthy,23,0
LPS,6,0


## Training and test patients

`train_test_split` splits the patients, so all cells of a patient land on the same side and the model is tested on
patients it has not seen. Stratifying by the label keeps its proportions in both sets.

In [4]:
train, test = hd.train_test_split(test_size=0.25, stratify="covid", random_state=0)
train.n_groups, test.n_groups

(89, 30)

## One bag of cells per patient

Every item of the dataset holds the patient features (`group_x`), the label (`group_y`) and the cells (`cell_x`). With
`n_cells`, every bag is subsampled or zero padded to the same number of cells and `cell_mask` marks the real ones, so
`mil_collate_fn` stacks the bags of a batch into one tensor. Two thirds of the patients have COVID-19, so the
training batches draw both classes equally often with `balanced_weights`.

In [5]:
bags = {
    "group_layer": "clinical",
    "group_labels_key": "covid",
    "cell_layer": "X_scVI",
    "n_cells": 256,
    "random_state": 0,
}
train_set = MILDataset(train, **bags)
test_set = MILDataset(test, **bags)
{key: tuple(value.shape) for key, value in train_set[0].items() if isinstance(value, torch.Tensor)}

{'group_x': (8,), 'group_y': (), 'cell_x': (256, 10), 'cell_mask': (256,)}

In [6]:
sampler = WeightedRandomSampler(
    train_set.balanced_weights(), num_samples=len(train_set), generator=torch.Generator().manual_seed(0)
)
train_loader = DataLoader(train_set, batch_size=16, sampler=sampler, collate_fn=mil_collate_fn)
test_loader = DataLoader(test_set, batch_size=16, collate_fn=mil_collate_fn)
batch = next(iter(train_loader))
batch["group_x"].shape, batch["cell_x"].shape, batch["cell_mask"].shape, batch["group_y"].shape

(torch.Size([16, 8]),
 torch.Size([16, 256, 10]),
 torch.Size([16, 256]),
 torch.Size([16]))

## Training

`GroupMILModel` encodes the patient features with a linear layer and every cell with a small network, weighs the
cells by attention and classifies the patient from both. With two outputs it is trained with the cross-entropy loss;
one output fits a regression instead. The model is small and trains on the CPU in seconds.

In [7]:
pl.seed_everything(0)
model = GroupMILModel(
    n_input_group=hd.obsm["clinical"].shape[1], n_input_cell=adata.obsm["X_scVI"].shape[1], n_output=2
)
trainer = pl.Trainer(
    max_epochs=40,
    accelerator="cpu",
    logger=False,
    enable_checkpointing=False,
    enable_progress_bar=False,
    enable_model_summary=False,
)
trainer.fit(model, train_loader, test_loader)
trainer.callback_metrics

Seed set to 0


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


`Trainer.fit` stopped: `max_epochs=40` reached.


{'train_loss': tensor(0.0940), 'val_loss': tensor(0.3722)}

## Predictions for the test patients

In [8]:
model.eval()
with torch.no_grad():
    batches = list(test_loader)
    p_covid = torch.cat([model(b).softmax(-1)[:, 1] for b in batches]).numpy()
patients = [patient for b in batches for patient in b["group_id"]]
predictions = pd.DataFrame(
    {"Status": test.G.loc[patients, "Status"], "covid": test.G.loc[patients, "covid"], "p_covid": p_covid},
    index=patients,
)
predictions["predicted"] = (predictions["p_covid"] > 0.5).astype(int)
pd.crosstab(predictions["Status"], predictions["predicted"])

predicted,0,1
Status,,
Covid,5,18
Healthy,6,0
LPS,1,0


The bags can also be built per sample: `MILDataset(hd, level="subgroup")` yields one bag per subgroup, for example
per visit of a patient, with the label and features of the subgroup level.